In [42]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [43]:
exp_name = "estimate1107"
node_name = "route_220"
es_scale = 180000
max_demand_price = 40.8

In [44]:
es_info = {"transform_capacity": 8883000,
            "invertband": 0,
            "soc_redundant_ratio": 0,
            "usable_depth": 0.90,
            "charge_loss": 0.92,
            "discharge_loss": 0.95,
            "es_charge_max": es_scale,
            "es_charge_min": -es_scale,
            "es_capacity_max": es_scale * 2,
            "es_capacity_min": 0}

In [45]:
def split_load_by_month(df):
    """
    将以时间戳为索引、包含'value'列的DataFrame按月拆分，返回一个字典。
    
    参数:
        df (pd.DataFrame): 索引为时间对象（datetime-like），包含'value'列。
    
    返回:
        dict: 键为'YYYY-MM'格式的字符串，值为对应月份的'value' Series。
    """
    # 确保索引是 datetime 类型
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index)
    
    # 按月分组
    monthly_groups = df.groupby(df.index.to_period('M'))
    
    # 构建字典：key 为 'YYYY-MM' 字符串，value 为该月的 'value' Series
    result = {
        str(period): group
        for period, group in monthly_groups
    }
    
    return result

def get_days_in_month(date_str):
    try:
        year, month = map(int, date_str.split('-'))
        # calendar.monthrange(year, month) 返回 (weekday_of_first_day, number_of_days)
        _, days = calendar.monthrange(year, month)
        return days
    except ValueError as e:
        print(f"输入格式错误或无效日期: {e}")
        return None

In [46]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/es_scale_experiment/schedule_result_scale_{es_scale}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [47]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)

In [48]:
month_load_dict = split_load_by_month(es_charge_df)

In [49]:
equivalent_charge_list = []
equivalent_discharge_list = []
for k,v in month_load_dict.items():
    days = get_days_in_month(k)
    count1 = (v['value'] > 0).sum() / days
    count2 = (v['value'] < 0).sum() / days
    count3 = v.loc[v['value'] > 0, 'value'].sum() / es_scale / days
    count4 = -v.loc[v['value'] < 0, 'value'].sum() / es_scale / days
    print(k, f"实际放电小时数:{count1}, 实际充电小时数:{count2}, 等效放电小时数:{count3}, 等效充电小时数:{count4}")
    equivalent_discharge_list.append(count3 / 4 * days)
    equivalent_charge_list.append(count4 / 4 * days)
equivalent_charge_days = sum(equivalent_charge_list)
equivalent_discharge_days = sum(equivalent_discharge_list)
print(f"年等效充电天数:{equivalent_charge_days}, 年等效放电天数:{equivalent_discharge_days}",)

2024-10 实际放电小时数:5.0, 实际充电小时数:14.96774193548387, 等效放电小时数:3.4199989793906815, 等效充电小时数:3.913042346398956
2024-11 实际放电小时数:5.0, 实际充电小时数:14.966666666666667, 等效放电小时数:3.4199991637037046, 等效充电小时数:3.913042509341469
2024-12 实际放电小时数:10.903225806451612, 实际充电小时数:8.96774193548387, 等效放电小时数:1.9881944378243728, 等效充电小时数:2.2748219872444086
2025-01 实际放电小时数:3.032258064516129, 实际充电小时数:8.96774193548387, 等效放电小时数:1.9881939028673832, 等效充电小时数:2.274821398944416
2025-02 实际放电小时数:5.0, 实际充电小时数:14.964285714285714, 等效放电小时数:3.419999946619445, 等效充电小时数:3.9130434178857465
2025-03 实际放电小时数:5.0, 实际充电小时数:14.96774193548387, 等效放电小时数:3.419999919713262, 等效充电小时数:3.913043389475324
2025-04 实际放电小时数:5.0, 实际充电小时数:14.966666666666667, 等效放电小时数:3.4199997074074076, 等效充电小时数:3.9130431549097375
2025-05 实际放电小时数:5.0, 实际充电小时数:14.96774193548387, 等效放电小时数:3.419999905017921, 等效充电小时数:3.913043359146346
2025-06 实际放电小时数:5.0, 实际充电小时数:14.966666666666667, 等效放电小时数:3.419999850925926, 等效充电小时数:3.913043307695567
2025-07 实际放电小时数:4.967741935483871, 实际充电小时数:9.0, 等效放电

In [50]:
type(v)

pandas.core.frame.DataFrame